# Train the model

In [39]:
import os
os.getcwd()

'/Users/charlie/Documents/Coding/VS Code/Language_python/FYP/New_2026/pyHGT-implementation/Test_recomentation'

In [40]:
import sys
import os

PROJECT_ROOT = os.path.abspath("/Users/charlie/Documents/Coding/VS Code/Language_python/FYP/New_2026/pyHGT-implementation")
sys.path.insert(0, PROJECT_ROOT)


Training Script for Test Recommendation System

This script trains a link prediction model to recommend medical tests for patients.
It extends the existing disease prediction model by adding a test recommendation component.

Key Features:
1. Loads the existing heterogeneous graph with patients, tests, organs, and diseases
2. Trains both disease prediction AND link prediction simultaneously
3. Link prediction learns patient-test associations
4. Evaluates recommendation quality using ranking metrics
5. Saves the trained model for test recommendations

Training Strategy:
- For each patient, we have their disease labels and tests they took
- We treat the patient-test edges as positive examples
- We sample negative examples (tests the patient didn't take)
- Train the Matcher to distinguish between useful and non-useful tests

In [41]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
import json

import sys
import torch
import torch.nn as nn
import torch.optim as optim
import sklearn.metrics
from sklearn.metrics import roc_auc_score, average_precision_score

from pyHGT.data import Graph, sample_subgraph, to_torch
from pyHGT.model import GNN, Matcher
from test_recommender import TestRecommender

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set seeds for reproducibility
import random
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================
# LOAD DATA (same as train.ipynb - UPDATED)
# ============================================

data_dir = '../data/'  # Go up one level from Test_recomentation/

# Use the correct data files matching train.ipynb
patient_tests = pd.read_csv(os.path.join(data_dir, "filtered_patient_reports.csv"), encoding='latin1')
patient_tests.drop(columns=['date_of_birth'], inplace=True, errors='ignore')

test_details = pd.read_csv(os.path.join(data_dir, "test-disease-organ.csv"), encoding='latin1')
labels_df = pd.read_csv(os.path.join(data_dir, "patient-one-hot-labeled-disease-new.csv"), encoding='latin1')

print(f"Loaded {len(patient_tests)} patient test records")
print(f"Loaded {len(test_details)} test details")
print(f"Loaded labels for {len(labels_df)} patients")

patient_tests["patient_id"] = patient_tests["patient_id"].astype(str)
patient_tests["test_name"] = patient_tests["test_name"].astype(str)

test_details["test_name"] = test_details["test_name"].astype(str)

if "organ" in test_details.columns:
    test_details["organ"] = test_details["organ"].astype(str)

if "disease" in test_details.columns:
    test_details["disease"] = test_details["disease"].astype(str)

test_details = test_details.drop_duplicates(subset=["test_name"]).reset_index(drop=True)

# Temporal processing
patient_tests["report_date"] = pd.to_datetime(patient_tests["report_date"])
patient_tests["time_idx"] = patient_tests["report_date"].astype("int64") // 10**9
min_time = patient_tests["time_idx"].min()
patient_tests["rel_time"] = (patient_tests["time_idx"]-min_time)// 86400

max_rel_time = int(patient_tests["rel_time"].max())
print("max_rel_time =", max_rel_time)

def parse_multi(x):
    if pd.isna(x):
        return []
    x = str(x).strip()
    if ";" in x:
        return [item.strip() for item in x.split(";")]
    if "," in x:
        return [item.strip() for item in x.split(",")]
    return [x] if x else []

# Build lab info
lab_info = {}
for _, row in test_details.iterrows():
    test = str(row["test_name"]).strip()
    organs = parse_multi(row.get("organ", ""))
    diseases = parse_multi(row.get("disease", ""))
    
    low_th = None
    high_th = None
    if "min" in row and not pd.isna(row["min"]):
        low_th = float(row["min"])
    if "max" in row and not pd.isna(row["max"]):
        high_th = float(row["max"])
    
    lab_info[test] = {
        "organs": organs,
        "diseases": diseases,
        "low": low_th,
        "high": high_th
    }

print("Number of unique tests:", len(lab_info))

# Compute abnormality features
pt_merged = patient_tests.merge(
    test_details[["test_name", "min", "max"]],
    on="test_name",
    how="left"
)

def compute_abnoramability(row):
    v = float(row["test_value"])
    low = row["min"]
    high = row["max"]
    if pd.isna(low) or pd.isna(high) or low >= high:
        return 0.0
    return float((v - low) / (high - low))

pt_merged["abnormality"] = pt_merged.apply(compute_abnoramability, axis=1)

# Aggregate patient features including age, sex, is_foreign
agg = pt_merged.groupby("patient_id").agg(
    num_tests = ("test_name", "count"),
    mean_abn = ("abnormality", "mean"),
    max_abn = ("abnormality", "max"),
    min_abn = ("abnormality", "min"),
    last_time = ("rel_time", "max"),
    age_at_report = ("age_at_report", "first"),  # Get age
    sex = ("sex", "first"),  # Get sex
    is_foreign = ("is_foreign", "first")  # Get is_foreign flag
).reset_index()

# Normalize age (matching train.ipynb)
max_age = agg["age_at_report"].max()
agg["age_normalized"] = agg["age_at_report"] / max_age if max_age > 0 else 0.0

# Encode sex (Male=1, Female=0)
agg["sex_encoded"] = agg["sex"].apply(lambda x: 1.0 if str(x).strip().lower() == 'male' else 0.0)

# Handle is_foreign (already 0/1)
agg["is_foreign"] = agg["is_foreign"].fillna(0).astype(float)

patient_feat_df = agg.set_index("patient_id")

print("\nPatient features include:")
print("  - num_tests, mean_abn, max_abn, min_abn, last_time")
print("  - age_normalized, sex_encoded, is_foreign")
print(f"  Total: 8 features per patient")

# Extract unique nodes
patient_ids = sorted(patient_tests["patient_id"].astype(str).unique().tolist())
lab_tests = sorted(list(lab_info.keys()))
all_organs = sorted({org for info in lab_info.values() for org in info["organs"] if org})
all_diseases = sorted({dis for info in lab_info.values() for dis in info["diseases"] if dis})

print("#patients =", len(patient_ids))
print("#labs =", len(lab_tests))
print("#organs =", len(all_organs))
print("#diseases =", len(all_diseases))

# ============================================
# BUILD GRAPH (same as train.ipynb)
# ============================================

graph = Graph()

# Add nodes with 8 features for patients
for pid in patient_ids:
    if pid in patient_feat_df.index:
        row = patient_feat_df.loc[pid]
        node = {
            "type": "patient",
            "id": pid,
            "num_tests": float(row["num_tests"]),
            "mean_abn":  float(row["mean_abn"]),
            "max_abn":   float(row["max_abn"]),
            "min_abn":   float(row["min_abn"]),
            "last_time": int(row["last_time"]),
            "age_normalized": float(row["age_normalized"]),
            "sex_encoded": float(row["sex_encoded"]),
            "is_foreign": float(row["is_foreign"]),
            "time":      int(row["last_time"]),
        }
    else:
        node = {
            "type": "patient",
            "id": pid,
            "num_tests": 0.0,
            "mean_abn":  0.0,
            "max_abn":   0.0,
            "min_abn":   0.0,
            "last_time": 0,
            "age_normalized": 0.0,
            "sex_encoded": 0.0,
            "is_foreign": 0.0,
            "time":      0,
        }
    graph.add_node(node)

for test in lab_tests:
    info = lab_info[test]
    graph.add_node({
        "type": "lab_test",
        "id": test,
        "time": 0,
        "low": 0.0 if info["low"] is None else float(info["low"]),
        "high": 0.0 if info["high"] is None else float(info["high"])
    })

for organ in all_organs:
    graph.add_node({
        "type": "organ",
        "id": organ,
        "time": 0
    })

for disease in all_diseases:
    graph.add_node({
        "type": "disease",
        "id": disease,
        "time": 0
    })

patient2idx = graph.node_forward["patient"]
lab2idx     = graph.node_forward["lab_test"]
organ2idx   = graph.node_forward["organ"]
disease2idx = graph.node_forward["disease"]

# Add edges
for _, row in patient_tests.iterrows():
    pid = str(row["patient_id"])
    test = str(row["test_name"])
    t = int(row["rel_time"])
    
    if pid in patient2idx and test in lab2idx:
        graph.add_edge(
            {"type": "patient", "id": pid},
            {"type": "lab_test", "id": test},
            time=t,
            relation_type="had_test",
            directed=True,
        )

for test, info in lab_info.items():
    for organ in info["organs"]:
        if organ in organ2idx:
            graph.add_edge(
                {"type": "lab_test", "id": test},
                {"type": "organ", "id": organ},
                relation_type="tests_organ",
                directed=True,
                time=0
            )
    
    for disease in info["diseases"]:
        if disease in disease2idx:
            graph.add_edge(
                {"type": "lab_test", "id": test},
                {"type": "disease", "id": disease},
                relation_type="associated_with",
                directed=True,
                time=0
            )
            
            for organ in info["organs"]:
                if organ in organ2idx:
                    graph.add_edge(
                        {"type": "disease", "id": disease},
                        {"type": "organ", "id": organ},
                        relation_type="occurs_in",
                        directed=True,
                        time=0
                    )

print("Meta relations in graph:", graph.get_meta_graph())

# Node features
for t, node_list in graph.node_bacward.items():
    df = pd.DataFrame(node_list).reset_index(drop=True)
    graph.node_feature[t] = df

for t,df in graph.node_feature.items():
    print(f"Node type: {t}, feature shape: {df.shape}")

def feature_medical_enriched(layer_data, graph):
    """
    Feature extraction function matching train.ipynb
    Patient nodes: 8 features (num_tests, mean_abn, max_abn, min_abn, last_time, age_normalized, sex_encoded, is_foreign)
    Lab test nodes: 5 features (low, high, 0, 0, 0)
    Other nodes: 5 features (all zeros)
    """
    feature = {}
    times = {}
    indxs = {}
    texts = {}
    
    all_types = ["patient", "lab_test", "organ", "disease"]
    
    for t in all_types:
        if t not in layer_data or len(layer_data[t]) == 0:
            # Adjust feature dimension for patients (8 features) vs others (5 features)
            n_feat = 8 if t == "patient" else 5
            feature[t] = np.zeros((0, n_feat), dtype=np.float32)
            times[t] = np.array((0,), dtype=np.int32)
            indxs[t] = np.array((0,), dtype=np.int32)
            continue
        
        g_idxs = np.array(list(layer_data[t].keys()), dtype=np.int64)
        tims = np.array(list(layer_data[t].values()))[:,1]
        
        df = graph.node_feature[t].reset_index(drop=True)
        
        # Set feature dimension based on node type
        n_feat = 8 if t == "patient" else 5
        feats = np.zeros((len(g_idxs), n_feat), dtype=np.float32)
        
        for i, g_idx in enumerate(g_idxs):
            local_idx = int(g_idx)
            
            if local_idx < len(df):
                if t == "patient":
                    # Extract all 8 features for patient nodes
                    cols = ["num_tests", "mean_abn", "max_abn", "min_abn", "last_time",
                            "age_normalized", "sex_encoded", "is_foreign"]
                    vals = df.iloc[local_idx][cols].astype(float).fillna(0.0).values.astype(np.float32)
                    feats[i] = vals
                elif t == "lab_test":
                    vals = df.iloc[local_idx][["low", "high"]].astype(float).fillna(0.0).values.astype(np.float32)
                    feats[i,:2] = vals  # first 2 features, rest are zeros
            else:
                # Index out of bounds, set to zeros
                feats[i] = 0.0
        
        feature[t] = feats
        times[t] = tims
        indxs[t] = g_idxs
    
    return feature, times, indxs, texts

print("\n✓ Feature extractor defined (patient: 8-dim, others: 5-dim)")

# Disease labels from the NEW label file
disease_label_cols = [c for c in labels_df.columns if c != "patient_id"]
num_diseases = len(disease_label_cols)

print("\nDisease labels from patient-one-hot-labeled-disease-new.csv:")
print("First 10 diseases:", disease_label_cols[:10])
print("Number of diseases to predict:", num_diseases)

num_patients_in_graph = len(patient2idx)
Y = torch.zeros((num_patients_in_graph, num_diseases), dtype=torch.float32)

for _, row in labels_df.iterrows():
    pid = str(row["patient_id"])
    if pid in patient2idx:
        idx = patient2idx[pid]
        Y[idx] = torch.tensor(row[disease_label_cols].values, dtype=torch.float32)

print("Labels tensor shape:", Y.shape)

# ============================================
# PREPARE LINK PREDICTION DATA
# ============================================

print("\n" + "="*50)
print("PREPARING LINK PREDICTION DATA")
print("="*50)

# Build patient-test ground truth
patient_test_pairs = []
for _, row in patient_tests.iterrows():
    pid = str(row["patient_id"])
    test = str(row["test_name"])
    if pid in patient2idx and test in lab2idx:
        patient_test_pairs.append((patient2idx[pid], lab2idx[test]))

# Remove duplicates
patient_test_pairs = list(set(patient_test_pairs))
print(f"Total positive patient-test pairs: {len(patient_test_pairs)}")

# Split into train/val/test for link prediction
np.random.shuffle(patient_test_pairs)

n_pairs = len(patient_test_pairs)
n_train_links = int(0.7 * n_pairs)
n_val_links = int(0.15 * n_pairs)

train_links = patient_test_pairs[:n_train_links]
val_links = patient_test_pairs[n_train_links:n_train_links+n_val_links]
test_links = patient_test_pairs[n_train_links+n_val_links:]

print(f"Train links: {len(train_links)}")
print(f"Val links: {len(val_links)}")
print(f"Test links: {len(test_links)}")

# Create negative samples (patient-test pairs that don't exist)
def sample_negative_links(positive_links, num_patients, num_tests, num_negatives, seed=42):
    """Sample negative links that don't exist in positive set"""
    np.random.seed(seed)  # Fixed seed for reproducibility
    positive_set = set(positive_links)
    negative_links = []
    
    max_attempts = num_negatives * 10
    attempts = 0
    
    while len(negative_links) < num_negatives and attempts < max_attempts:
        pid = np.random.randint(0, num_patients)
        tid = np.random.randint(0, num_tests)
        if (pid, tid) not in positive_set and (pid, tid) not in negative_links:
            negative_links.append((pid, tid))
        attempts += 1
    
    return negative_links

num_patients = len(patient2idx)
num_tests = len(lab2idx)

# Sample negatives with fixed seeds for reproducibility
train_neg_links = sample_negative_links(train_links, num_patients, num_tests, len(train_links), seed=42)
val_neg_links = sample_negative_links(val_links, num_patients, num_tests, len(val_links), seed=43)
test_neg_links = sample_negative_links(test_links, num_patients, num_tests, len(test_links), seed=44)

print(f"Negative train links: {len(train_neg_links)}")
print(f"Negative val links: {len(val_neg_links)}")
print(f"Negative test links: {len(test_neg_links)}")

# ============================================
# MODEL INITIALIZATION
# ============================================

types = graph.get_types()
num_types = len(types)

meta_rels = graph.get_meta_graph()
num_relations = len(meta_rels) + 1

print("\nNode types:", types)
print("Number of node types:", num_types)
print("Meta relations:", meta_rels)
print("Number of relations:", num_relations)

# IMPORTANT: in_dim = 7 to match train.ipynb
# (Patient features are 8-dim, but GNN uses 7 as max across node types since others are 5-dim)
in_dim = 7
hidden_dim = 64
n_heads = 4
n_layers = 2
dropout = 0.2

print(f"\n✓ Model configuration: in_dim={in_dim}, hidden_dim={hidden_dim}, n_heads={n_heads}, n_layers={n_layers}")

gnn = GNN(
    in_dim=in_dim,
    n_hid=hidden_dim,
    num_types=num_types,
    num_relations=num_relations,
    n_heads=n_heads,
    n_layers=n_layers,
    dropout=dropout,
    conv_name='hgt',
    prev_norm=False,
    last_norm=False,
    use_RTE=True
).to(device)

# Disease classifier
class MultilabelClassifier(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
    
    def forward(self, x):
        return self.linear(x)

clf = MultilabelClassifier(hidden_dim, num_diseases).to(device)

# Link prediction matcher
matcher = Matcher(hidden_dim).to(device)

# Optimizer for all components
params = list(gnn.parameters()) + list(clf.parameters()) + list(matcher.parameters())
optimizer = optim.Adam(params, lr=0.001, weight_decay=1e-5)

# Loss functions
disease_criterion = nn.BCEWithLogitsLoss()
link_criterion = nn.BCEWithLogitsLoss()

# Test recommender
recommender = TestRecommender(matcher, confidence_threshold=0.7)

print("✓ Models initialized successfully")

# ============================================
# TRAINING FUNCTIONS
# ============================================

def get_embeddings_for_nodes(node_indices, node_type_name, graph, time_range, sampled_depth=2, sampled_number=8):
    """
    Get embeddings for specific nodes from the GNN.
    
    Args:
        node_indices: List of node indices
        node_type_name: Type of nodes (e.g., 'patient', 'lab_test')
        graph: The heterogeneous graph
        time_range: Time range for sampling
        sampled_depth: Depth of subgraph sampling
        sampled_number: Number of neighbors to sample
    
    Returns:
        embeddings: Tensor of embeddings [num_nodes, hidden_dim]
    """
    inp = {
        node_type_name: [(int(nid), 0) for nid in node_indices]
    }
    
    feature, times, edge_list, indxs, texts = sample_subgraph(
        graph,
        time_range=time_range,
        sampled_depth=sampled_depth,
        sampled_number=sampled_number,
        inp=inp,
        feature_extractor=feature_medical_enriched
    )
    
    node_feature, node_type, edge_time, edge_index, edge_type, node_dict, edge_dict = to_torch(
        feature, times, edge_list, graph
    )
    
    node_feature = node_feature.to(device)
    node_type = node_type.to(device)
    edge_time = edge_time.to(device)
    edge_index = edge_index.to(device)
    edge_type = edge_type.to(device)
    
    with torch.set_grad_enabled(True):
        all_embs = gnn(node_feature, node_type, edge_time, edge_index, edge_type)
    
    node_offset, node_type_id = node_dict[node_type_name]
    local_node_ids = indxs[node_type_name]
    
    # Map requested indices to local indices
    nid_to_local = {int(nid): i for i, nid in enumerate(local_node_ids)}
    
    selected_global_indices = []
    for nid in node_indices:
        if nid in nid_to_local:
            local_id = nid_to_local[nid]
            global_node_idx = node_offset + local_id
            selected_global_indices.append(global_node_idx)
    
    if len(selected_global_indices) == 0:
        return None
    
    selected_global_indices = torch.LongTensor(selected_global_indices).to(device)
    embeddings = all_embs[selected_global_indices]
    
    return embeddings

def train_link_prediction_batch(links, is_positive, batch_size=32):
    """
    Train on a batch of links for link prediction.
    
    Args:
        links: List of (patient_idx, test_idx) tuples
        is_positive: Boolean indicating if these are positive examples
        batch_size: Batch size for training
    
    Returns:
        loss: Average loss for this batch
    """
    if len(links) == 0:
        return 0.0
    
    # Randomly sample a batch
    indices = np.random.choice(len(links), min(batch_size, len(links)), replace=False)
    batch_links = [links[i] for i in indices]
    
    patient_indices = [link[0] for link in batch_links]
    test_indices = [link[1] for link in batch_links]
    
    # Get embeddings
    time_range = {max_rel_time: True}
    patient_embs = get_embeddings_for_nodes(patient_indices, "patient", graph, time_range)
    test_embs = get_embeddings_for_nodes(test_indices, "lab_test", graph, time_range)
    
    if patient_embs is None or test_embs is None:
        return 0.0
    
    # Compute link scores
    scores = matcher(patient_embs, test_embs, infer=False, pair=True)
    
    # Labels
    labels = torch.ones(len(batch_links), device=device) if is_positive else torch.zeros(len(batch_links), device=device)
    
    # Loss
    loss = link_criterion(scores, labels)
    
    return loss

def evaluate_link_prediction(links_pos, links_neg, batch_size=64):
    """
    Evaluate link prediction performance.
    
    Args:
        links_pos: Positive links
        links_neg: Negative links
        batch_size: Batch size
    
    Returns:
        metrics: Dict with AUC, AP, and accuracy
    """
    gnn.eval()
    matcher.eval()
    
    all_scores = []
    all_labels = []
    
    with torch.no_grad():
        # Evaluate positive links
        for i in range(0, len(links_pos), batch_size):
            batch = links_pos[i:i+batch_size]
            patient_indices = [link[0] for link in batch]
            test_indices = [link[1] for link in batch]
            
            time_range = {max_rel_time: True}
            patient_embs = get_embeddings_for_nodes(patient_indices, "patient", graph, time_range)
            test_embs = get_embeddings_for_nodes(test_indices, "lab_test", graph, time_range)
            
            if patient_embs is None or test_embs is None:
                continue
            
            scores = matcher(patient_embs, test_embs, infer=False, pair=True)
            scores = torch.sigmoid(scores).cpu().numpy()
            
            all_scores.extend(scores.tolist())
            all_labels.extend([1] * len(scores))
        
        # Evaluate negative links
        for i in range(0, len(links_neg), batch_size):
            batch = links_neg[i:i+batch_size]
            patient_indices = [link[0] for link in batch]
            test_indices = [link[1] for link in batch]
            
            time_range = {max_rel_time: True}
            patient_embs = get_embeddings_for_nodes(patient_indices, "patient", graph, time_range)
            test_embs = get_embeddings_for_nodes(test_indices, "lab_test", graph, time_range)
            
            if patient_embs is None or test_embs is None:
                continue
            
            scores = matcher(patient_embs, test_embs, infer=False, pair=True)
            scores = torch.sigmoid(scores).cpu().numpy()
            
            all_scores.extend(scores.tolist())
            all_labels.extend([0] * len(scores))
    
    if len(all_scores) == 0:
        return {"auc": 0.0, "ap": 0.0, "acc": 0.0}
    
    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    
    auc = roc_auc_score(all_labels, all_scores)
    ap = average_precision_score(all_labels, all_scores)
    acc = np.mean((all_scores > 0.5) == all_labels)
    
    return {"auc": auc, "ap": ap, "acc": acc}

print("✓ Training functions defined")
print("\nReady to train or load pre-trained model!")

Using device: cpu
Loaded 28167 patient test records
Loaded 182 test details
Loaded labels for 5766 patients
max_rel_time = 204
Number of unique tests: 172

Patient features include:
  - num_tests, mean_abn, max_abn, min_abn, last_time
  - age_normalized, sex_encoded, is_foreign
  Total: 8 features per patient
#patients = 5766
#labs = 172
#organs = 19
#diseases = 45
Meta relations in graph: [('lab_test', 'patient', 'had_test'), ('lab_test', 'organ', 'rev_tests_organ'), ('lab_test', 'disease', 'rev_associated_with'), ('patient', 'lab_test', 'rev_had_test'), ('organ', 'lab_test', 'tests_organ'), ('organ', 'disease', 'occurs_in'), ('disease', 'lab_test', 'associated_with'), ('disease', 'organ', 'rev_occurs_in')]
Node type: patient, feature shape: (5766, 11)
Node type: lab_test, feature shape: (172, 5)
Node type: organ, feature shape: (19, 3)
Node type: disease, feature shape: (45, 3)

✓ Feature extractor defined (patient: 8-dim, others: 5-dim)

Disease labels from patient-one-hot-labeled-d

# recommend_tests

Test Recommendation Inference Script

This script demonstrates how to use the trained test recommendation model
to recommend tests for patients based on their current test results and
disease prediction confidence.

Usage:
    python recommend_tests.py --patient_id <patient_id>
    python recommend_tests.py --demo  # Run demo with sample patients


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import json

from pyHGT.data import Graph, sample_subgraph, to_torch
from pyHGT.model import GNN, Matcher, Classifier
from test_recommender import TestRecommender

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set seeds for reproducibility
import random
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ============================================
# LOAD TRAINED MODEL (from train.ipynb)
# ============================================

def load_pretrained_disease_model(model_path='models_saved/trained_model4.pth'):
    """
    Load the pre-trained disease prediction model from train.ipynb
    This model was trained with in_dim=5 (OLD architecture, no patient metadata)
    and 44 disease labels (OLD label file)
    """
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    print(f"✓ Loaded checkpoint from: {model_path}")
    print(f"  Available keys: {checkpoint.keys()}")
    
    return checkpoint

def load_test_recommendation_model(model_path='models_saved/test_recommendation_model.pth'):
    """
    Load the trained test recommendation model (link prediction)
    """
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    # Model parameters
    in_dim = checkpoint.get('in_dim', 7)
    hidden_dim = checkpoint['hidden_dim']
    num_types = checkpoint['num_types']
    num_relations = checkpoint['num_relations']
    n_heads = checkpoint.get('n_heads', 4)
    n_layers = checkpoint.get('n_layers', 2)
    num_diseases = checkpoint['num_diseases']
    
    # Initialize models
    gnn = GNN(
        in_dim=in_dim,
        n_hid=hidden_dim,
        num_types=num_types,
        num_relations=num_relations,
        n_heads=n_heads,
        n_layers=n_layers,
        dropout=0.2,
        conv_name='hgt',
        prev_norm=False,
        last_norm=False,
        use_RTE=True
    ).to(device)
    
    matcher = Matcher(hidden_dim).to(device)
    
    class MultilabelClassifier(torch.nn.Module):
        def __init__(self, in_dim, out_dim):
            super().__init__()
            self.linear = torch.nn.Linear(in_dim, out_dim)
        
        def forward(self, x):
            return self.linear(x)
    
    clf = MultilabelClassifier(hidden_dim, num_diseases).to(device)
    
    # Load weights
    gnn.load_state_dict(checkpoint['gnn_state_dict'])
    matcher.load_state_dict(checkpoint['matcher_state_dict'])
    clf.load_state_dict(checkpoint['clf_state_dict'])
    
    # Set to evaluation mode
    gnn.eval()
    matcher.eval()
    clf.eval()
    
    print(f"✓ Loaded test recommendation model")
    print(f"  in_dim={in_dim}, hidden_dim={hidden_dim}, num_diseases={num_diseases}")
    
    return gnn, matcher, clf, checkpoint

# ============================================
# LOAD DATA AND BUILD GRAPH (matching train.ipynb)
# ============================================

def load_graph_and_data():
    """Load the same graph structure as training"""
    data_dir = '../data/'  # Go up one level from Test_recomentation/
    
    patient_tests = pd.read_csv(os.path.join(data_dir, "filtered_patient_reports.csv"), encoding='latin1')
    patient_tests.drop(columns=['date_of_birth'], inplace=True, errors='ignore')
    
    test_details = pd.read_csv(os.path.join(data_dir, "test-disease-organ.csv"), encoding='latin1')
    
    # IMPORTANT: Use OLD label file (44 diseases) to match trained_model4.pth
    labels_df = pd.read_csv(os.path.join(data_dir, "patient-one-hot-labeled-disease.csv"), encoding='latin1')
    
    patient_tests["patient_id"] = patient_tests["patient_id"].astype(str)
    patient_tests["test_name"] = patient_tests["test_name"].astype(str)
    test_details["test_name"] = test_details["test_name"].astype(str)
    
    if "organ" in test_details.columns:
        test_details["organ"] = test_details["organ"].astype(str)
    if "disease" in test_details.columns:
        test_details["disease"] = test_details["disease"].astype(str)
    
    test_details = test_details.drop_duplicates(subset=["test_name"]).reset_index(drop=True)
    
    # Temporal processing
    patient_tests["report_date"] = pd.to_datetime(patient_tests["report_date"])
    patient_tests["time_idx"] = patient_tests["report_date"].astype("int64") // 10**9
    min_time = patient_tests["time_idx"].min()
    patient_tests["rel_time"] = (patient_tests["time_idx"]-min_time)// 86400
    max_rel_time = int(patient_tests["rel_time"].max())
    
    def parse_multi(x):
        if pd.isna(x):
            return []
        x = str(x).strip()
        if ";" in x:
            return [item.strip() for item in x.split(";")]
        if "," in x:
            return [item.strip() for item in x.split(",")]
        return [x] if x else []
    
    # Build lab info
    lab_info = {}
    for _, row in test_details.iterrows():
        test = str(row["test_name"]).strip()
        organs = parse_multi(row.get("organ", ""))
        diseases = parse_multi(row.get("disease", ""))
        
        low_th = None
        high_th = None
        if "min" in row and not pd.isna(row["min"]):
            low_th = float(row["min"])
        if "max" in row and not pd.isna(row["max"]):
            high_th = float(row["max"])
        
        lab_info[test] = {
            "organs": organs,
            "diseases": diseases,
            "low": low_th,
            "high": high_th
        }
    
    # Compute features (OLD architecture - 5 dimensions only)
    pt_merged = patient_tests.merge(
        test_details[["test_name", "min", "max"]],
        on="test_name",
        how="left"
    )
    
    def compute_abnoramability(row):
        v = float(row["test_value"])
        low = row["min"]
        high = row["max"]
        if pd.isna(low) or pd.isna(high) or low >= high:
            return 0.0
        return float((v - low) / (high - low))
    
    pt_merged["abnormality"] = pt_merged.apply(compute_abnoramability, axis=1)
    
    agg = pt_merged.groupby("patient_id").agg(
        num_tests = ("test_name", "count"),
        mean_abn = ("abnormality", "mean"),
        max_abn = ("abnormality", "max"),
        min_abn = ("abnormality", "min"),
        last_time = ("rel_time", "max")
    ).reset_index()
    
    patient_feat_df = agg.set_index("patient_id")
    
    # Extract unique nodes
    patient_ids = sorted(patient_tests["patient_id"].astype(str).unique().tolist())
    lab_tests = sorted(list(lab_info.keys()))
    all_organs = sorted({org for info in lab_info.values() for org in info["organs"] if org})
    all_diseases = sorted({dis for info in lab_info.values() for dis in info["diseases"] if dis})
    
    # Build graph
    graph = Graph()
    
    for pid in patient_ids:
        if pid in patient_feat_df.index:
            row = patient_feat_df.loc[pid]
            node = {
                "type": "patient",
                "id": pid,
                "num_tests": float(row["num_tests"]),
                "mean_abn":  float(row["mean_abn"]),
                "max_abn":   float(row["max_abn"]),
                "min_abn":   float(row["min_abn"]),
                "last_time": int(row["last_time"]),
                "time":      int(row["last_time"]),
            }
        else:
            node = {
                "type": "patient",
                "id": pid,
                "num_tests": 0.0,
                "mean_abn":  0.0,
                "max_abn":   0.0,
                "min_abn":   0.0,
                "last_time": 0,
                "time":      0,
            }
        graph.add_node(node)
    
    for test in lab_tests:
        info = lab_info[test]
        graph.add_node({
            "type": "lab_test",
            "id": test,
            "time": 0,
            "low": 0.0 if info["low"] is None else float(info["low"]),
            "high": 0.0 if info["high"] is None else float(info["high"])
        })
    
    for organ in all_organs:
        graph.add_node({
            "type": "organ",
            "id": organ,
            "time": 0
        })
    
    for disease in all_diseases:
        graph.add_node({
            "type": "disease",
            "id": disease,
            "time": 0
        })
    
    patient2idx = graph.node_forward["patient"]
    lab2idx     = graph.node_forward["lab_test"]
    organ2idx   = graph.node_forward["organ"]
    disease2idx = graph.node_forward["disease"]
    
    # Add edges
    for _, row in patient_tests.iterrows():
        pid = str(row["patient_id"])
        test = str(row["test_name"])
        t = int(row["rel_time"])
        
        if pid in patient2idx and test in lab2idx:
            graph.add_edge(
                {"type": "patient", "id": pid},
                {"type": "lab_test", "id": test},
                time=t,
                relation_type="had_test",
                directed=True,
            )
    
    for test, info in lab_info.items():
        for organ in info["organs"]:
            if organ in organ2idx:
                graph.add_edge(
                    {"type": "lab_test", "id": test},
                    {"type": "organ", "id": organ},
                    relation_type="tests_organ",
                    directed=True,
                    time=0
                )
        
        for disease in info["diseases"]:
            if disease in disease2idx:
                graph.add_edge(
                    {"type": "lab_test", "id": test},
                    {"type": "disease", "id": disease},
                    relation_type="associated_with",
                    directed=True,
                    time=0
                )
                
                for organ in info["organs"]:
                    if organ in organ2idx:
                        graph.add_edge(
                            {"type": "disease", "id": disease},
                            {"type": "organ", "id": organ},
                            relation_type="occurs_in",
                            directed=True,
                            time=0
                        )
    
    # Node features
    for t, node_list in graph.node_bacward.items():
        df = pd.DataFrame(node_list).reset_index(drop=True)
        graph.node_feature[t] = df
    
    # Disease labels (44 diseases from OLD label file)
    disease_label_cols = [c for c in labels_df.columns if c != "patient_id"]
    
    return graph, patient_tests, labels_df, disease_label_cols, max_rel_time

def feature_medical(layer_data, graph):
    """
    Feature extraction function for OLD model (5-dimensional features)
    Patient nodes: 5 features (num_tests, mean_abn, max_abn, min_abn, last_time)
    Lab test nodes: 5 features (low, high, 0, 0, 0)
    Other nodes: 5 features (all zeros)
    """
    feature = {}
    times = {}
    indxs = {}
    texts = {}
    
    for t in layer_data:
        if len(layer_data[t]) == 0:
            continue
        
        idxs = np.array(list(layer_data[t].keys()))
        tims = np.array(list(layer_data[t].values()))[:,1]
        
        df = graph.node_feature[t]
        feats = np.zeros((len(idxs),5), dtype=np.float32)
        
        if t == "patient":
            cols = ["num_tests", "mean_abn", "max_abn", "min_abn", "last_time"]
            vals = df.loc[idxs, cols].fillna(0).values.astype(np.float32)
            feats = vals
        elif t == "lab_test":
            for c in ["low", "high"]:
                if c not in df.columns:
                    df[c] = 0.0
            vals = df.loc[idxs, ["low", "high"]].fillna(0).values.astype(np.float32)
            feats[:,0:2] = vals
        else:
            pass
        
        feature[t] = feats
        times[t] = tims
        indxs[t] = idxs
    
    return feature, times, indxs, texts

# ============================================
# RECOMMENDATION FUNCTION
# ============================================

def recommend_tests_for_patient(patient_id, gnn, matcher, clf, graph, disease_label_cols, max_rel_time, top_k=5):
    """
    Recommend tests for a specific patient.
    
    Args:
        patient_id: Patient ID (string)
        gnn: Trained GNN model
        matcher: Trained Matcher model
        clf: Trained classifier
        graph: Heterogeneous graph
        disease_label_cols: List of disease labels
        max_rel_time: Maximum relative time
        top_k: Number of tests to recommend
    
    Returns:
        recommendations: Dict with patient info and recommendations
        disease_predictions: Disease prediction results
    """
    patient2idx = graph.node_forward["patient"]
    lab2idx = graph.node_forward["lab_test"]
    
    if str(patient_id) not in patient2idx:
        return None, None
    
    patient_idx = patient2idx[str(patient_id)]
    
    # Get patient embedding
    inp = {"patient": [(int(patient_idx), 0)]}
    
    feature, times, edge_list, indxs, texts = sample_subgraph(
        graph,
        time_range={max_rel_time: True},
        sampled_depth=2,
        sampled_number=8,
        inp=inp,
        feature_extractor=feature_medical
    )
    
    node_feature, node_type, edge_time, edge_index, edge_type, node_dict, edge_dict = to_torch(
        feature, times, edge_list, graph
    )
    
    node_feature = node_feature.to(device)
    node_type = node_type.to(device)
    edge_time = edge_time.to(device)
    edge_index = edge_index.to(device)
    edge_type = edge_type.to(device)
    
    with torch.no_grad():
        all_embs = gnn(node_feature, node_type, edge_time, edge_index, edge_type)
    
    patient_offset, _ = node_dict["patient"]
    patient_emb = all_embs[patient_offset:patient_offset+1]
    
    # Disease prediction
    disease_logits = clf(patient_emb)
    disease_probs = torch.sigmoid(disease_logits)
    
    # Get all test embeddings
    all_test_indices = list(lab2idx.values())
    inp_tests = {"lab_test": [(int(tid), 0) for tid in all_test_indices]}
    
    feature_t, times_t, edge_list_t, indxs_t, texts_t = sample_subgraph(
        graph,
        time_range={max_rel_time: True},
        sampled_depth=2,
        sampled_number=8,
        inp=inp_tests,
        feature_extractor=feature_medical
    )
    
    node_feature_t, node_type_t, edge_time_t, edge_index_t, edge_type_t, node_dict_t, edge_dict_t = to_torch(
        feature_t, times_t, edge_list_t, graph
    )
    
    node_feature_t = node_feature_t.to(device)
    node_type_t = node_type_t.to(device)
    edge_time_t = edge_time_t.to(device)
    edge_index_t = edge_index_t.to(device)
    edge_type_t = edge_type_t.to(device)
    
    with torch.no_grad():
        all_embs_t = gnn(node_feature_t, node_type_t, edge_time_t, edge_index_t, edge_type_t)
    
    test_offset, _ = node_dict_t["lab_test"]
    test_embs = all_embs_t[test_offset:test_offset+len(all_test_indices)]
    
    # Get recommendations
    recommender = TestRecommender(matcher, confidence_threshold=0.7)
    recommendations = recommender.recommend_tests(
        patient_emb,
        test_embs,
        [patient_id],
        disease_probs,
        graph,
        top_k=top_k,
        force_recommend=False
    )
    
    # Format disease predictions
    disease_pred_dict = {}
    for i, disease in enumerate(disease_label_cols):
        prob = disease_probs[0, i].item()
        if prob > 0.3:  # Only show diseases with reasonable probability
            disease_pred_dict[disease] = prob
    
    # Sort by probability
    disease_pred_dict = dict(sorted(disease_pred_dict.items(), key=lambda x: x[1], reverse=True))
    
    return recommendations[0], disease_pred_dict

print("✓ Functions loaded successfully!")
print("✓ Using OLD architecture (5-dim features, no patient metadata)")
print("✓ Using OLD label file (44 diseases) to match trained_model4.pth")
print("✓ Ready to load model and make recommendations")

Using device: cpu
✓ Functions loaded successfully!
✓ Using OLD architecture (5-dim features, no patient metadata)
✓ Ready to load model and make recommendations


### Load Model and Data

Run this cell to load the trained model and build the graph. This needs to be done once before making recommendations.

**Two options:**
1. **Use pre-trained disease model** from train.ipynb (recommended) - loads GNN + Classifier
2. **Train new test recommendation model** - trains link prediction from scratch

If you have a trained test recommendation model, it will load that. Otherwise, it will use the disease prediction model and you can train link prediction separately.

In [44]:
# Load the graph and data
print("Loading graph data...")
graph, patient_tests, labels_df, disease_label_cols, max_rel_time = load_graph_and_data()

print(f"✓ Graph loaded: {len(graph.node_bacward['patient'])} patients, {len(graph.node_bacward['lab_test'])} tests")
print(f"✓ Disease labels: {len(disease_label_cols)} diseases")
print(f"  Diseases: {disease_label_cols[:5]}...")

# Load the pre-trained model (trained_model4.pth uses OLD architecture)
model_path = '../models_saved/trained_model4.pth'
checkpoint = load_pretrained_disease_model(model_path)

# Model parameters (matching trained_model4.pth)
in_dim = 5  # OLD architecture: only 5 features (no age/sex/is_foreign)
hidden_dim = 256
num_types = len(graph.get_types())
num_relations = len(graph.get_meta_graph()) + 1
n_heads = 8
n_layers = 2
num_diseases = len(disease_label_cols)

print(f"\nInitializing model with:")
print(f"  in_dim={in_dim} (OLD: num_tests, mean_abn, max_abn, min_abn, last_time)")
print(f"  hidden_dim={hidden_dim}")
print(f"  n_heads={n_heads}")
print(f"  n_layers={n_layers}")
print(f"  num_diseases={num_diseases}")

# Initialize GNN
gnn = GNN(
    in_dim=in_dim,
    n_hid=hidden_dim,
    num_types=num_types,
    num_relations=num_relations,
    n_heads=n_heads,
    n_layers=n_layers,
    dropout=0.2,
    conv_name='hgt',
    prev_norm=False,
    last_norm=False,
    use_RTE=True
).to(device)

# Initialize Classifier  
class MultilabelClassifier(torch.nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = torch.nn.Linear(in_dim, out_dim)
    
    def forward(self, x):
        return self.linear(x)

clf = MultilabelClassifier(hidden_dim, num_diseases).to(device)

# Load weights from checkpoint
gnn.load_state_dict(checkpoint['gnn_state_dict'])
clf.load_state_dict(checkpoint['clf_state_dict'])

# Set to evaluation mode
gnn.eval()
clf.eval()

# Create a dummy matcher (not used for disease prediction, only for test recommendation)
matcher = Matcher(hidden_dim).to(device)
matcher.eval()

print(f"\n✓ Model loaded successfully!")
print(f"✓ GNN and Classifier ready for disease prediction")
print(f"✓ Note: This model uses OLD architecture (5-dim features only)")
print(f"✓ To use patient metadata (age/sex/is_foreign), retrain with cell 5")

Loading graph data...
✓ Graph loaded: 5766 patients, 172 tests
✓ Disease labels: 26 diseases
  Diseases: ['acid base imbalances', 'adrenal gland disorders', 'anemia', 'artharities', 'biliary obstruction']...
✓ Loaded checkpoint from: ../models_saved/trained_model4.pth
  Available keys: dict_keys(['gnn_state_dict', 'clf_state_dict', 'optimizer_state_dict'])

Initializing model with:
  in_dim=5 (OLD: num_tests, mean_abn, max_abn, min_abn, last_time)
  hidden_dim=256
  n_heads=8
  n_layers=2
  num_diseases=26


RuntimeError: Error(s) in loading state_dict for MultilabelClassifier:
	size mismatch for linear.weight: copying a param with shape torch.Size([44, 256]) from checkpoint, the shape in current model is torch.Size([26, 256]).
	size mismatch for linear.bias: copying a param with shape torch.Size([44]) from checkpoint, the shape in current model is torch.Size([26]).

### Option 1: Recommend Tests for a Specific Patient

Specify a patient ID to get test recommendations.

In [ ]:
# Specify patient ID and number of recommendations
patient_id = "139760"  # Change this to any patient ID
top_k = 10  # Number of tests to recommend


print(f"Recommending tests for patient: {patient_id}")
print("="*70)

recommendations, disease_preds = recommend_tests_for_patient(
    patient_id, gnn, matcher, clf, graph, disease_label_cols, max_rel_time, top_k=top_k
)

if recommendations is None:
    print(f"Patient {patient_id} not found in graph.")
else:
    # Display disease predictions
    print("\n--- Disease Predictions (Top 10) ---")
    for i, (disease, prob) in enumerate(list(disease_preds.items())[:10]):
        print(f"  {i+1}. {disease}: {prob:.3f}")
    
    # Display recommendations
    print(f"\n--- Test Recommendations ---")
    print(f"Confidence: {recommendations['confidence']:.3f}")
    print(f"Mean Confidence: {recommendations['mean_confidence']:.3f}")
    print(f"Entropy (uncertainty): {recommendations['entropy']:.3f}")
    print(f"Needs recommendation: {recommendations['needs_recommendation']}")
    
    if recommendations['needs_recommendation']:
        print(f"\nRecommended Tests:")
        for test_rec in recommendations['recommended_tests']:
            print(f"  {test_rec['rank']}. {test_rec['test_name']} (score: {test_rec['score']:.3f})")
    else:
        print("\n✓ High confidence - no additional tests needed at this time.")

Recommending tests for patient: 139760

--- Disease Predictions (Top 10) ---
  1. kidney damage: 0.622
  2. allergies: 0.600
  3. cancer: 0.584
  4. hyperlipidaemia: 0.577
  5. thyroid disorders: 0.573
  6. adrenal gland disorders: 0.561
  7. leukemia: 0.555
  8. anemia: 0.547
  9. cardiovascular diseases: 0.538
  10. biliary obstruction: 0.537

--- Test Recommendations ---
Confidence: 0.622
Mean Confidence: 0.498
Entropy (uncertainty): 0.685
Needs recommendation: True

Recommended Tests:
  1. Triglycerides New Value (score: 0.985)
  2. nan (score: 0.971)
  3. Direct Bilirubin value (score: 0.952)
  4. Monocytes# Absolute Value (score: 0.944)
  5. S.G.P.T (ALT) Value (score: 0.942)
  6. GGT Value (score: 0.939)
  7. Alkaline Phosphatase Value (score: 0.916)
  8. WBC % Value (score: 0.912)
  9. Total Bilirubin value (score: 0.877)
  10. Neutrophils# % Value (score: 0.860)


### Option 2: Demo with Random Patients

Get recommendations for multiple random patients to see how the system works.

In [ ]:
# Demo with random patients
num_demo_patients = 5  # Change this to see more/fewer patients
top_k = 5

all_patient_ids = list(graph.node_forward["patient"].keys())
sample_patient_ids = np.random.choice(all_patient_ids, min(num_demo_patients, len(all_patient_ids)), replace=False)

print("="*70)
print("DEMO: Test Recommendations for Sample Patients")
print("="*70)

for pid in sample_patient_ids:
    print(f"\n{'='*70}")
    print(f"Patient ID: {pid}")
    print(f"{'='*70}")
    
    recommendations, disease_preds = recommend_tests_for_patient(
        pid, gnn, matcher, clf, graph, disease_label_cols, max_rel_time, top_k=top_k
    )
    
    if recommendations is None:
        print(f"Patient {pid} not found in graph.")
        continue
    
    # Display disease predictions
    print("\n--- Disease Predictions (Top 5) ---")
    for i, (disease, prob) in enumerate(list(disease_preds.items())[:5]):
        print(f"  {i+1}. {disease}: {prob:.3f}")
    
    # Display recommendations
    print(f"\n--- Test Recommendations ---")
    print(f"Confidence: {recommendations['confidence']:.3f}")
    print(f"Needs recommendation: {recommendations['needs_recommendation']}")
    
    if recommendations['needs_recommendation']:
        print(f"\nRecommended Tests:")
        for test_rec in recommendations['recommended_tests']:
            print(f"  {test_rec['rank']}. {test_rec['test_name']} (score: {test_rec['score']:.3f})")
    else:
        print("\n✓ High confidence - no additional tests needed.")

DEMO: Test Recommendations for Sample Patients

Patient ID: 1174052

--- Disease Predictions (Top 5) ---
  1. infections: 0.819
  2. cancer: 0.819
  3. kidney damage: 0.794
  4. heart attacks: 0.739
  5. kidney stones: 0.721

--- Test Recommendations ---
Confidence: 0.819
Needs recommendation: False

✓ High confidence - no additional tests needed.

Patient ID: 507283

--- Disease Predictions (Top 5) ---
  1. cancer: 0.824
  2. fluid: 0.814
  3. cholestrol: 0.790
  4. infections: 0.775
  5. leukemia: 0.770

--- Test Recommendations ---
Confidence: 0.824
Needs recommendation: False

✓ High confidence - no additional tests needed.

Patient ID: 1290771

--- Disease Predictions (Top 5) ---
  1. fluid: 0.843
  2. liver damage: 0.813
  3. infections: 0.801
  4. cancer: 0.799
  5. leukemia: 0.748

--- Test Recommendations ---
Confidence: 0.843
Needs recommendation: False

✓ High confidence - no additional tests needed.

Patient ID: 608256

--- Disease Predictions (Top 5) ---
  1. hemolytic ane